# Instal·lació de llibreries

In [2]:
#Install the necessary Python libraries
# import requests
#Install the necessary Python libraries
%pip install requests==2.32.4 -q --force-reinstall
%pip install langchain_community 
%pip install faiss-cpu -q
%pip install openai -q 
%pip install python-dotenv -q 
%pip install sentence_transformers -q 
%pip install torch -q 
%pip install transformers -q 

%pip install -q langchain
%pip install -U langchain-classic
%pip install -q langchain-huggingface
%pip install -q langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# make the necessary imports

In [11]:
#from langchain.document_loaders import TextLoader
#from langchain.vectorstores import FAISS
#from langchain.embeddings import HuggingFaceEmbeddings
#from langchain.text_splitter import RecursiveCharacterTextSplitter

#from langchain.chains import RetrievalQA

#from langchain.llms import HuggingFacePipeline
#from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub' (c:\Users\daniel.herrero.ext\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\__init__.py)

# Load the document

In [4]:
def load_document(path):
    loader = TextLoader(path, encoding="utf-8")
    documents = loader.load()
    return documents

# Create embeddings with thenlper/gte-small

In [5]:
def prepare_knowledge_base(documents):
    # Split documents to create the chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    # Create chunk embeddings
    embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")

    # Initialize FAISS vector store
    knowledge_base = FAISS.from_documents(chunks, embedding_model)
    len(chunks)
    return knowledge_base

# Create the LLM model sshleifer/tiny-gpt2

In [6]:
def create_llm():
    model_id = "sshleifer/tiny-gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)

    # Set pad token to avoid index out of range errors
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    llm_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100,
        temperature=1.0,
        top_k=50,
        device=0 if torch.cuda.is_available() else -1,
    )
    return HuggingFacePipeline(pipeline=llm_pipeline)

# Create the QA chain

In [7]:
def build_chain(knowledge_base, llm):
    # Create the retrieval-based QA chain
    retriever = knowledge_base.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    return chain

# Terminal interface

In [8]:
def terminal_interface(qa_chain):
    print("System for questions about the document. Type 'exit' to finish.\n")
    while True:
        pregunta = input("make a question: ")
        if pregunta.lower() == "exit":
            break
        resposta = qa_chain.invoke(pregunta)
        print(f"\nanswer: {resposta['result']}\n")

# Execució General

In [9]:
if __name__ == "__main__":
    document_path = "Spider-Man.txt"

    if not os.path.exists(document_path):
        print(f"File '{document_path}' not found.")
    else:
        documents = load_document(document_path) # Load the document
        knowledge_base = prepare_knowledge_base(documents) # Split and index document
        llm = create_llm() # Create the language model
        chain = build_chain(knowledge_base, llm) # Build the QA chain
        terminal_interface(chain) # Start the terminal interface

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.